# Target Grouping and Identification

This notebook is intended to plan for Target identification and grouping.
Identification is based on a combination of DICOM Structure Type: 
*RT ROI Interpreted Type (3006,00A4)*
and the Structure ID:
*ROI Name (3006,0026)*

- The RT ROI Interpreted Type values that are defined as Targets are specified.

- Regular expressions are defined that can be used to identify target structures and to group them.

Test examples are Use to demonstrate the functionality of the identification and grouping process.

## Setup

### Imports

In [1]:
from typing import Dict, List, Union

from pathlib import Path
import re

from pprint import pprint

In [2]:
import xml.etree.ElementTree as ET
from itertools import chain


In [3]:
import pandas as pd
import xlwings as xw

In [4]:
from structure_set import StructureSet
from dicom import DicomStructureFile

INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Paths

In [5]:
base_path = Path.cwd()
dicom_path = base_path / "tests"
structure_filter_def = base_path / "src" / "webapp" / "config" / "structure_filter_rules.json"

## Load an example H&N DICOM structure set

### Path to the example DICOM file

In [70]:
dcm_file_path = dicom_path / "RS.HN_Struct.OROP.dcm"
#dcm_file_path = dicom_path / 'RS.CNS_FSRT_2_GTV.BRAI.dcm'
#dcm_file_path = dicom_path / 'RS.4Field_Logic.BRER.dcm'
#dcm_file_path = dicom_path / 'RS.LUNR_Test.LUNR.dcm'

### Load the DICOM file

In [71]:
dicom_file = DicomStructureFile(top_dir=dicom_path, file_path=dcm_file_path)
pprint(dicom_file.structure_set_info)

INFO:dicom:Successfully loaded DICOM dataset from RS.HN_Struct.OROP.dcm


INFO:dicom:Extracted 3666 contours from 61 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.HN_Struct.OROP.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.070 cm/pixel


{'File': WindowsPath("d:/OneDrive - Queen's University/Python/Projects/StructureRelations/tests/RS.HN_Struct.OROP.dcm"),
 'PatientID': 'HN_Struct',
 'PatientLastName': 'Head',
 'PatientName': 'Head^and^Neck^Structures',
 'SeriesDescription': 'ARIA RadOnc Structure Sets',
 'SeriesNumber': '234',
 'StructureSet': 'OROP',
 'StudyID': ''}


### Filter unwanted structures

In [72]:
filter_report = dicom_file.evaluate_structure_filters(structure_filter_def)
selection = filter_report.SelectedByDefault & filter_report.DisplayedByDefault
meta_data = dicom_file.get_structure_filter_metadata()[selection]
include_structures = list(meta_data['Structure ID'])

In [19]:
exclude_structures = [
    '[$ZzD].*', 'BODY', 'Avoid.*', '.*Bolus.*', '.*Brain.*', 'Cochlea.*',
    'Esophagus', 'Globe.*', '.*Larynx.*', 'Lens.*', 'Mandible', '.*Optic.*',
    '.*Parotid.*', '.*PRV.*', 'SpinalCanal', 'Submandibular.*', 'Brachial.*',
    'Lip', 'Musc.*', 'Oral.*'
    ]


### Determine the relationships between the structures

## Target Nomenclature

In [62]:
# Initialize dictionary with descriptions for each regex group.
group_definitions = {}
# Initialize list to hold regex sections.
regex_sections = []

### Target Modifier Prefixes
- `eval` Target volume explicitly intended for DVH evaluation
- `opt` Target volume only intended for optimization. 
- `mod` Target volume modified for unspecified reasons.
- `shell` Spherical shell surrounding a target volume used for optimizing dose fall-off. 

In [63]:
target_mod_def = {
    'eval': 'Target volume explicitly intended for DVH evaluation',
    'opt': 'Target volume only intended for optimization',
    'mod': 'Target volume modified for unspecified reasons',
    'shell': 'Spherical shell surrounding a target volume used for optimizing dose fall-off'
    }
# Add the target type definitions to the group_definitions dictionary.
group_definitions['Mod'] = target_mod_def

target_mod = '|'.join(target_mod_def.keys())
target_mod_pat = ''.join([
    r'(?P<Mod>',      # Start of *Mod* group
    f'{target_mod}',    # contains '|' separated items from target_mod_def.
    r')?',            # End of optional *Mod* group
    r'[ _]*'          # Optional space or '_'
    ])

regex_sections.append(target_mod_pat)

### Target Type

- The first set of characters must be one of the allowed target types:
    - GTV
    - CTV
    - ITV
    - IGTV (Internal Gross Target Volume—gross disease with margin for motion)
    - ICTV (Internal Clinical Target Volume—clinical disease with margin for motion)
    - PTV
    - PTV! for low-dose PTV volumes that exclude overlapping high-dose volumes

- Additional Target types not defined in TG-263:
  - Nodes, Node:  Used to identify a nodal volume that is not a GTV, but used to define a CTV or   PTV.
  - Iliac Vessels Special case of pelvic vessels used as a surrogate for pelvic nodes.
  - Edema: Used as a contrasted-based indicator of CTV volumes in CNS tumours.
  - Cavity: Used in breast cancer, where the GTV is defined as the surgical cavity.
  - Operative Bed: Used for post-surgery cases where the target is the operative bed.
  - HTV: High-risk target volume.
  - HRV: High-risk volume.

In [64]:
target_type_def = {
    'GTV': 'Gross Target Volume',
    'CTV': 'Clinical Target Volume',
    'PTV': 'Planning Target Volume',
    'ITV': 'Internal Target Volume',
    'IGTV': 'Internal Gross Target Volume',
    'ICTV': 'Internal Clinical Target Volume',
    'PTV!': 'Partial Planning Target Volume',
    # Additional local target types not defined in TG-263:
    'Nodes': 'Nodal Volume',
    'Node': 'Nodal Volume',
    'Iliac Vessels': 'Nodal Volume',
    'Edema': 'CTV Surrogate',
    'Cavity': 'GTV Surrogate',
    'Operative Bed': 'GTV Surrogate',
    'HTV': 'Clinical Target Volume',
    'HRV': 'Clinical Target Volume',
    }

# Add the target type definitions to the group_definitions dictionary.
group_definitions['TargetType'] = target_type_def

# Create a regular expression pattern to match the target types defined in target_type_def.
target_type = '|'.join(target_type_def.keys())
target_type_pat = ''.join([
    r'(?P<TargetType>',  # Start of required *TargetType* group
    f'{target_type}',    # contains '|' separated items from target_type.
    r')'                 # End of *TargetType* group.
    ])

regex_sections.append(target_type_pat)

### Target Classifier

- If used, the target classifier is placed after the target type with no spaces.
    - Allowed target classifiers are listed below: 
    - n: nodal (e.g., PTVn) 
    - p: primary (e.g., GTVp) 
    - m: metastatic (e.g., CTVm) *Non TG-263 target classifier*
    - sb: surgical bed (e.g., CTVsb) 
    - par: parenchyma (e.g., GTVpar) 
    - v:venous thrombosis (e.g., CTVv) 
    - vas: vascular (e.g., CTVvas)

- Other non TG-263 target classifiers are also used:
    - low: Low Dose Target
    - int: Intermediate Dose Target
    - IR: Intermediate Risk Target Volume
    - HR: High Risk Target Volume
    - PREOP: Pre-operative Target Volume
    - RES: Residual Disease Target Volume
    - Cavity: Target Expansion on the Cavity

In [ ]:
target_classifier_def = {
    'n': 'nodal',
    'p': 'primary',
    'm': 'metastatic',
    'sb': 'surgical bed',
    'par': 'parenchyma',
    'v': 'venous thrombosis',
    'vas': 'vascular',
    'low': 'Low Dose Target',
    'int': 'Intermediate Dose Target',
    'IR': 'Intermediate Risk Target Volume',
    'HR': 'High Risk Target Volume',
    'PREOP': 'Pre-operative Target Volume',
    'RES': 'Residual Disease Target Volume',
    'Cavity': 'Target Expansion on the Cavity'
    }

# Add the target type definitions to the group_definitions dictionary.
group_definitions['Classifier'] = target_classifier_def

# Create a regular expression pattern to match the target types defined in target_classifier_def.
target_classifier = '|'.join(target_classifier_def.keys())
# Checks on characters following the classifier to ensure that it is not part
# of a longer word.  For example, 'n' should not match 'node'.
target_classifier_pat = ''.join([
    r'(?:'                   # Start of an optional non-capturing group.
    r'(?P<Classifier>',        # Start of required *Classifier* group
    f'{target_classifier}',      # contains '|' separated items from target_type.
    r')'                      # End of the optional *Classifier* group.
    r'(?![D-Zd-z]{{2}})'      # NOT followed by any two characters except for a, b, or c.
    r')?'                 # End of the optional non-capturing group.
    ])

regex_sections.append(target_classifier_pat)

### Target Number

- For multiple spatially distinct targets Arabic numerals are used after the 
  target type and classifier (e.g., PTV1, PTV2, GTVp1, GTVp2).

The regular expression is more complex than most to ensure that the target 
number is not part of a larger number, such as a dose value.

In [66]:
# Create a regular expression pattern to match the target number as 1 or 2 digits.
# A group definition dictionary is not required for the target number because
# it is just a number.

# Characters preceding and following the group also need to be tested to ensure
# that it is a target number and not a dose value or expansion number. The target
# number is not allowed to be proceeded by a digit, '.', '+', or '-'. It is also
# not allowed to be followed by a digit, '.', 'D', 'c', or 'm'.

target_number_pat = ''.join([
    r'(?:',                # Start of an optional non-capturing group.
    r'[ _]*',                # Optional space or '_'
    r'(?<![0-9.+-])',        # NOT preceded by another digit, '.', '+', or '-'.
    r'(?P<TargetNumber>',    # Start of *TargetNumber* group
    r'[1-9]',                  # A single digit.
    r'|',                      # OR
    r'1[0-5]',                 # A '1' and an additional digit between 0 and 4.
    r')'                     # End of the *TargetNumber* group.
    r'(?![0-9.Dcm])',        # NOT followed by a digit, '.', 'D', 'c', or 'm'.
    r')?',                   # End of the optional non-capturing group.
    ])

regex_sections.append(target_number_pat)

### Image Modality Designator

- Imaging modality follows the type/classifier/enumerator with an underscore 
  and then the image modality type (CT, PT, MR, SP)

- 'T' is also included as an abbreviation for MR T1, or 'T2'.  This is not a TG-263 designation.

- Image sequence order is indicated by a number immediately following the 
  image modality.

- Unlike the TG-263 protocol, multiple modalities cannot be included. 

In [67]:
target_modality_def = {
    'CT': 'CT',
    'PT': 'PET',
    'PET': 'PET',
    'MR': 'MRI',
    'MRI': 'MRI',
    'T': 'MRI',
    'US': 'Ultrasound',
    'SP': 'SPECT'
    }

# Add the target modality definitions to the group_definitions dictionary.
group_definitions['Modality'] = target_classifier_def

# Create a regular expression pattern to match the target types defined in target_modality_def.
target_modality = '|'.join(target_modality_def.keys())
target_modality_pat = ''.join([
    r'[ _]*',               # Optional space or '_'
   r'(?:',                  # Start of an optional non-capturing group.
    r'(?P<Modality>',         # Start of named group *Modality*
    f'{target_modality}',       # contains '|' separated items from target_modality.
    r')',                     # End of named group *Modality*
    r'(?P<ModalityNumber>',   # Start of optional named group *ModalityNumber*
    r'[0-9]{0,2}',              # Optional sequence number as 1 or 2 digits
    r')?',                    # End of optional named group *ModalityNumber*
    r')?'                   # End of optional group
    ])

regex_sections.append(target_modality_pat)

### Motion Management Designator

For 4D scanning target modifiers are available to indicate both aggregates and phase bins

- Aggregated *4D* target motion includes the following designators:
  - **MIP**: Maximum intensity projection, 
  - **AIP**: or *AVE*: Average intensity projection,
  - **MinIP**:  Minimum intensity projection,
  - **4D##**: Target volume on a particular motion phase, 
      where ## is the phase number (usually 10, 20, ...)


In [68]:
target_motion_def = {
    'MIP': 'Maximum intensity projection',
    'AIP': 'Average intensity projection',
    'AVE': 'Average intensity projection',
    'MinIP': 'Minimum intensity projection'
    }

# Add the target motion definitions to the group_definitions dictionary.
group_definitions['Motion'] = target_motion_def

# Create a regular expression pattern to match the target motion options.
# The named motion option and the 4D phase groups are supplied as seperate
# optional groups for simplicity.  Based on the regex expression, it as possible
# for both an aggregate and a phase indicator to coexist e.g. 'PTV Ave 4D10'.
# However this is not a logical combination and we assume that no one would
# actually combine them in a target name.
target_motion = '|'.join(target_motion_def.keys())
target_motion_pat = ''.join([
    r'[ _]*',             # Optional space or '_'
    r'(?P<Motion>',         # Start of optional named group *Motion*
    f'{target_motion}',     # contains '|' separated items from target_motion.
    r')?',                # End of optional named group *Motion*
    r'(?P<MotionPhase>',  # Start of optional named group *MotionPhase*
    r'4D[0-9][0-9]?'        # '4D' followed by one or two digits.
    r')?',                # End of optional named group *MotionPhase*
    ])

regex_sections.append(target_motion_pat)

# Done To Here

In [69]:
final_regex = re.compile(''.join(regex_sections))
for structure in include_structures:
    match = final_regex.match(structure)
    if match:
        groups = match.groupdict()
        pprint(f"Structure: {structure}, Groups: {groups}")
    else:
        pprint(f"Structure: {structure} did not match the regex.")

("Structure: GTV PET, Groups: {'Mod': None, 'TargetType': 'GTV', 'Classifier': "
 "None, 'TargetNumber': None, 'Modality': 'PET', 'ModalityNumber': '', "
 "'Motion': None, 'MotionPhase': None}")
("Structure: IGTV, Groups: {'Mod': None, 'TargetType': 'IGTV', 'Classifier': "
 "None, 'TargetNumber': None, 'Modality': None, 'ModalityNumber': None, "
 "'Motion': None, 'MotionPhase': None}")
("Structure: ITV, Groups: {'Mod': None, 'TargetType': 'ITV', 'Classifier': "
 "None, 'TargetNumber': None, 'Modality': None, 'ModalityNumber': None, "
 "'Motion': None, 'MotionPhase': None}")
("Structure: PTV, Groups: {'Mod': None, 'TargetType': 'PTV', 'Classifier': "
 "None, 'TargetNumber': None, 'Modality': None, 'ModalityNumber': None, "
 "'Motion': None, 'MotionPhase': None}")
("Structure: opt ITV RT, Groups: {'Mod': 'opt', 'TargetType': 'ITV', "
 "'Classifier': None, 'TargetNumber': None, 'Modality': None, "
 "'ModalityNumber': None, 'Motion': None, 'MotionPhase': None}")
("Structure: opt ITV LT, Gr

##### Numeric Dose
- Units of cGy are recommended for numeric dose values. (e.g., PTV_5040).
- When specified in units of Gy, then ‘Gy’ should be appended to the numeric 
  value of the dose (e.g., PTV_50.4Gy). 
- For systems that do not allow use of a period, the ‘p’ character should be 
  substituted (e.g., PTV_50p4Gy)

In [ ]:
numeric_dose_pat = ''.join([
    r'(?P<NumericDose>',  # Start of optional named group NumericDose
    r'[0-9]+',              # Number before decimal place
    r'[.p]?',               # '.' or 'p' as optional decimal place
    r'[0-9]*',              # Optional Number after decimal place
    r'[Gy]*',               # Optional units of Gy
    r')'                  # End of NumericDose group
    ])


##### Dose per Fraction and number of Fractions
- If the dose indicated must reflect the number of fractions used to reach the 
  total dose, then the numeric values of <u>dose per fraction</u> in cGy, or 
  in Gy with the <u>unit specifier</u>, '<u>x</u>' followed by the <u>number 
  of fractions</u> (e.g., PTV_Liver_2000x3 or PTV_Liver_20Gyx3).

In [ ]:
dose_fraction_pat = ''.join([
    r'(?P<DoseFractionation>',  # Start of  named group DoseFractionation
    r'[0-9]+',                    # Number before decimal place
    r'[.p]?',                     # '.' or 'p' as optional decimal place
    r'[0-9]*',                    # Optional Number after decimal place
    r'[Gy]*',                     # Optional units of Gy
    r'x',                         # Fractions delimiter 'x'
    r'[0-9]+',                    # Number of fractions
    r')'                        # End of DoseFractionation group
    ])


- Dose Fractionation must be specified before Numeric Dose because otherwise Numeric Dose will catch part of Dose Fractionation

In [ ]:
dose_specifier = ''.join([
   '(',                   # Beginning of optional Dose Specifier group
    r'(?:_)',               # Underscore delimiter (Not captured)
    r'(?P<DoseSpecifier>',  # Start of named group DoseSpecifier
    rel_dose_pat,             # Relative Dose pattern definition
    '|',                    # OR
    dose_fraction_pat,        # Dose Fractionation pattern definition
    '|',                    # OR
    numeric_dose_pat,         # Numeric Dose pattern definition
    ')'                     # End of Dose Specifier options
    ')?',                 # End of optional Dose Specifier group
   ])


In [ ]:
    # Dose (Optional Dose Value modifier)
    target_pattern_list.append(
        # Pattern optionally ends with units that are not captured.
        # '-' is allowed as a delimiter because subtracting dose doesn't happen.
        # The double { and } brackets are used to 'escape' { and } when
        # applying str.format().
        r'(?:'                 # Start of an optional non-capturing group.
        r'(?:[- _]*)'          # Optional non-capturing blanks (space '_' or '-').
        r'(?P<TargetDose>'           # Start of the *TargetDose* group
        r'[0-9.]{{2,}}'        # Two or more digits (includes decimal)
        r')'                   # End of the *TargetDose* group
        r'(?:[ cGy]{{2,3}})?'  # Optional non-capturing units: cGy or Gy.
        # Optional non-capturing fractions '/' followed by number and optional '#'
        r'(?:/[0-9]+#?)?'
        r')?'                  # End of the optional non-capturing group
        r'[ _]*'               # Optional space or '_'
        )


### Targets Combined
- `_Total` as the combined structure for all targets at the same dose level.

In [ ]:

    # Combined
    target_group_options['combined_types'] = [
        'Total', 'All']
    target_pattern_list.append(
        r'(?:'               # Start of an optional non-capturing group.
        r'(?:[- ]*)'         # non-capturing optional minus ('-') with spaces.
        r'(?P<Combined>'     # Start of the *Combined* group
        r'{combined_types}'  # contains '|' separated items from combined_types.
        r')'                 # End of the *Combined* group
        r')?'                # End of the optional non-capturing group
        r'[ _]*'             # Optional space or '_'
        )


In [ ]:

    # TargetOrgan
    target_pattern_list.append(
        r'(?P<TargetOrgan>'      # Start of the optional *TargetOrgan* group
        r'[A-Za-z]{{3,}}'  # Name of an organ. (Must be longer than 3 letters.)
        r')?'              # End of the optional *TargetOrgan* group
        r'[ _]*'           # Optional space or '_'
        )


### Target Expansion
PTVs are sometimes expanded to evaluate the conformality of a plan.
While not mentioned in the text, the supplied TG263 examples suggest the 
following format:
- `_Ev##` (where `##` is the uniform expansion size in mm)


In [ ]:

    # Expansions
    target_pattern_list.append(
        r'(?:'                  # Start of an optional non-capturing group.
        #  First option number followed by units
        r'(?P<ExpansionSize1>'  # Start of the *ExpansionSize1* group
        r'[0-9.]+'              # Number (one or more digits including decimal)
        r')'                    # End of the *ExpansionSize1* group
        r'(?P<ExpansionUnit1>'  # Start of the *ExpansionUnit1* group
        r'[cm]m'                # *mm* or *cm*
        r')'                    # End of the *ExpansionUnit1* group
        #  End of First option
        r'|'                    # OR
        #  Second option '+' or '-' followed by number with optional units
        r'(?P<ExpansionSign>'   # Start of the *ExpansionSign* group
        r'[+-])'                # Plus or minus sign
        r'(?P<ExpansionSize2>'  # Start of the *ExpansionSize2* group
        r'[0-9.]+'              # Number (one or more digits including decimal)
        r')'                    # End of the *ExpansionSize2* group
        r'(?P<ExpansionUnit2>'  # Start of the optional *ExpansionUnit2* group
        r'[cm]m'                # *mm* or *cm*
        r')?'                   # End of the optional *ExpansionUnit2* group
        #  End of Second option
        r')?'                   # End of the optional non-capturing group
        r'[ _]*'                # Optional space or '_'
        )


### Target Organ Subtraction
There are times when modified targets are generated by subtracting a critical 
OAR from the initial target volume
- `-<Organ>`  (Where `<Organ>` represents and valid OAR structure name)

In [ ]:

    # OrganSubtraction
    target_pattern_list.append(
        # Pattern starts with '-' that is not captured.
        r'(?:'                    # Start of an optional non-capturing group.
        r'(?P<SubtractionSign>'   # Start of the *SubtractionSign* group
        r'-'                      # Minus sign ('-')
        r')'                      # End of the *Subtraction* group.
        r' *'                     # Optional white space.
        r'(?P<OrganSubtraction>'  # Start of the *OrganSubtraction* group
        r'[A-Za-z]*'              # OAR label without spaces or numbers.
        r')'                      # End of the *OrganSubtraction* group.
        r')?'                     # End of the optional non-capturing group.
        r'[ _]*'                  # Optional space or '_'
        )


In [ ]:

    # Laterality
    target_group_options['laterality_types'] = ['L', 'LT', 'R', 'RT']
    target_pattern_list.append(
        r'(?P<TargetLaterality>'     # Start of the optional *TargetLaterality* group
        r'{laterality_types}'  # contains '|' separated items from Level_types.
        r')?'                  # End of the optional *TargetLaterality* group
        r'[ _]*'               # Optional space or '_'
        )


### Target Subgroup
For optimization purposes a target volume may be divided into subsections 
based on the proximity to hight dose targets.  The TG263 document implies that 
such structures should be prefixed with a `z` or an `_`.

An alternative would be to append a letter suffix to the full target volume to 
designate that it is a subsection.
- `~a`, `~b` `~c`

The `~` delimiter would have the same meaning of *partial structure* that it 
has for OAR structures.


In [ ]:

    # SubGroup
    target_pattern_list.append(
        r'(?P<SubGroup>'  # Start of the optional *SubGroup* group
        r'[abc]'          # One of 'a', 'b', or 'c'
        r')?'             # End of the optional *SubGroup* group
        )


- If the structure is cropped back from the external contour for the patient, 
  then the quantity of cropping by “-xx” millimeters is placed at the end of 
  the target string. The cropping length follows the dose indicator, with the 
  amount of cropping indicated by xx millimeters 
  (e.g., PTV_Eval_7000-08, PTV-03, CTVp2-05).

In [ ]:
target_crop_pat = ''.join([
    r'(?P<ExternalCrop>',  # Start of optional named group ExternalCrop
    r'(?P<Sign>-)',          # Negative sign '-' as its own named group
    r'(?P<Size>[0-9]{2})',   # 2-digit Number
    r')?'                  # End of optional group
    ])


In [ ]:
    # Remaining Text
    target_pattern_list.append(
        r'(?P<Remainder>'  # Start of the optional *Remainder* group
        r'.+'              # All non-captured text after the Target Type
        r')?'              # End of the optional *Remainder* group
        )
     # End of string
    target_pattern_list.append(r'$')

#### Base Target
- Base target include first three parts of target structure name:
  1. Target Type
  2. Target Classifier
  3. Target Number

- The base target is grouped separately because it can be used as a cropping 
  designator for OARs. e.g. `Brain-GTV`

In [ ]:
base_target_pat = ''.join([
    r'(?P<BaseTarget>',      # Start of named group BaseTarget
    r'(?P<TargetType>',        # Start of named group TargetType
    target_type_pat,             # Target Type pattern definition
    r')',                      # End of the TargetType group
    r'(?P<TargetClassifier>',  # Start of optional named group TargetClassifier
    target_classifier_pat,       # Target Classifier pattern definition
    r')?',                     # End of the optional TargetClassifier group
    r'(?P<TargetNumber>',      # Start of optional named group TargetNumber
    target_number_pat,           # Target Number pattern definition
    r')?',                     # End of the optional TargetNumber group
    r')',                    # End of the BaseTarget group
    ])


#### Dose Specifier

- Dose specifier is placed at the end of the target string prefixed with an underscore character.

- Dose specifier can be one of:
   - Relative Dose Level
   - Numeric dose
   - Dose per Fraction and number of Fractions


##### Relative Dose
- Relative dose is recommended
    - High (e.g., PTV_High, CTV_High, GTV_High)
    - Mid: (e.g., PTV_Mid, CTV_Mid, GTV_Mid)
    - Low (e.g., PTV_Low, CTV_Low, GTV_Low) ◦ 

- Mid+2-digit enumerator: allows specification of more than three relative 
  dose levels (e.g., PTV_Low, PTV_Mid01, PTV_Mid02, PTV_Mid03, PTV_High). 
- Lower numbers correspond to lower dose values.

In [ ]:
rel_dose_pat = ''.join([
    r'(?P<RelativeDose>',     # Start of named group RelativeDose
    r'High|',                   # 'High' relative dose    OR
    r'Low|',                    # 'Low'  relative dose    OR
    r'Mid',                     # 'Mid'  relative dose   with
    r'(?P<RelativeDoseLevel>',  # Optional RelativeDoseLevel group
    r'[0-9]{2}',                  # 2-digit number
    r')?',                      # End of Optional RelativeDoseLevel group
    r')'                      # End of RelativeDose group
    ])


#### Combine Target patterns
**Order of Name Components:**  (Underline indicates required component)

1. <u>Base Target</u>  (consists of three parts)
   1. <u>Target Type</u>
   2. Target Classifier
   3. Target Number
2. Modality
3. Structure Indicator
4. Dose Specifier
5. Target Cropping from External
6.  Custom Qualifier

- Pattern must match the entire string


### Get Dose Values

In [ ]:
dose_values = matched_structures.DoseSpecifier.apply(to_cgy)
matched_structures = matched_structures.join(dose_values)


### Identify match failures

In [ ]:
unmatched_idx = matched_structures.isna().all(axis='columns')
unmatched_idx.name = 'Unmatched'
matched_structures = matched_structures.join(unmatched_idx)


### Utility Functions

In [ ]:
def combine_columns(df: pd.DataFrame, columns: List[str], sep=' ')->pd.Series:
    '''Combine text from multiple columns with a separator.

    Args:
        df (pd.DataFrame): The table containing the columns to be merged.
            All of the columns should contain strings or NA values.
        columns (List[str]: The names of the columns to be merged.
        sep (str): A delimiter to place between the text from each column.

    Returns:
        pd.Series: A new text column containing the combined text from each
            column.
    '''
    row_dict = {}
    for index, row in df.iterrows():
        text_items = []
        for col in columns:
            text = row.at[col]
            if text:
                text_items.append(str(text))
        combined_text = sep.join(text_items)
        row_dict[index] = combined_text
    new_col = pd.Series(row_dict)
    return new_col


In [ ]:
def to_cgy(text: str)->Dict[str, Union[float, int]]:
    '''Convert numbers with Gy units to cGy and identify fractions.

    Numbers without units or x are assumed to be total dose in cGy.
    Numbers with trailing 'Gy' are converted to cGy.
    Number preceded by x are fractions. dose values are assumed to be dose
    per fraction. Total dose is calculated by:
        $dose_per_fraction x fractions$
    The decimal point may also be represented by a 'p' e.g. 50p4Gy
    If the text does not match any of the valid formats, return the original
    text.

    Args:
        text (str): Dose as a string in one of the following forms:
            ####
            ##Gy
            ##.##Gy
            ##p##Gy
            ####x#
            ##Gyx#
            ##.##Gyx#
            ##p##Gyx#

    Returns:
        Tuple[float, int]: _description_
    '''
    if not text:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Convert 'p' to decimal point.
    try:
        text_cnv1 = text.replace('p', '.')
    except AttributeError:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Find fractions
    dose_parts = text_cnv1.split('x', 1)
    if len(dose_parts) > 1:
        try:
            fractions = int(dose_parts[1])
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    else:
        fractions = None
    dose_str = dose_parts[0]
    # Convert Gy to cGy
    if dose_str.endswith('Gy'):
        try:
            dose = float(dose_str[:-2])  # Drop the Gy suffix
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
        dose = dose * 100   # Gy to cGy conversion
    else:
        try:
            dose = float(dose_str)
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    # Convert dose per fraction to total dose
    if fractions:
        total_dose = dose * fractions
    else:
        total_dose = dose
    dose_dict = {'TotalDose': total_dose, 'Fractions': fractions}
    return pd.Series(dose_dict)


In [ ]:
def extract_name_group(names: pd.DataFrame, re_pattern: re.Pattern,
                       match_column: str, idx: pd.Series = None)->pd.DataFrame:
    '''Extract portions of a structure name.

    The re_pattern is applied to the 'Remainder' column to extract named parts.
    The resulting DataFrame is merged with names and the Remainder columns is
    updated with new Remainders from the extraction.

    Args:
        nt_names (pd.DataFrame): A table with structure names. It must contain
            a column 'Remainder', which is used as the starting point for
            extracting name parts.
        re_pattern (re.Pattern): A regular expression with named groups.  It
            must contain one named group that will always be present if a
            successful match is made.  It must also contain a 'Remainder'
            named group that contains the unmatched part of the structure name.
        match_column (str): The name of the named group that is always present
            when a successful match is made.  Used to update the 'Remainder'
            column.
        idx (pd.Series, optional): A mask type index to a subset of names to
            apply the match to.  If not supplied, all names are matched.
            Default is None.

    Returns:
        pd.DataFrame: The supplied table with new columns containing the
            structure name parts.
    '''
    # Extract group parts based on regular expression.
    if idx is not None:
        extr_names = names.loc[idx, 'Remainder'].str.extract(re_pattern)
    else:
        extr_names = names.loc[:, 'Remainder'].str.extract(re_pattern)

    # Merge extracted group parts with structure names.
    names = names.merge(extr_names, how='left',
                            left_index=True, right_index=True,
                            suffixes=('', '_ex'))

    # Update Remainder text
    nt_idx = names[match_column].isna()
    # Where a match was not found, keep the original Remainder text, otherwise
    # update Remainder with resulting Remainder after the match.
    names.Remainder = names.Remainder.where(nt_idx, names.Remainder_ex)
    names.drop(columns=['Remainder_ex'], inplace=True)
    return names


**Structure Set ROI Sequence (3006,0020)**

- **ROI Number (3006,0022)**
    Identification number of the ROI. The value of *ROI Number (3006,0022)* 
    shall be unique within the Structure Set in which it is created.

- **ROI Name (3006,0026)**
    User-defined name for ROI.


**RT ROI Observations Sequence (3006,0080)**

- **Referenced ROI Number (3006,0084)**
    Uniquely identifies the referenced ROI described in the 
    *Structure Set ROI Sequence (3006,0020)*.

- **ROI Observation Label (3006,0085)**
    User-defined label for ROI Observation.

- **RT ROI Interpreted Type (3006,00A4)**
    Type of ROI.
  - Defined Terms:

|Term|Definition|
|----|----------|
|EXTERNAL|external patient contour|
|PTV|Planning Target Volume (as defined in ICRU50)|
|CTV|Clinical Target Volume (as defined in ICRU50)|
|GTV|Gross Tumor Volume (as defined in ICRU50)|
|TREATED_VOLUME|Treated Volume (as defined in ICRU50)|
|IRRAD_VOLUME|Irradiated Volume (as defined in ICRU50)|
|BOLUS|patient bolus to be used for external beam therapy|
|AVOIDANCE|region in which dose is to be minimized|
|ORGAN|patient organ|
|MARKER|patient marker or marker on a localizer|
|REGISTRATION|registration ROI|
|ISOCENTER|treatment isocenter to be used for external beam therapy|
|CONTRAST_AGENT|volume into which a contrast agent has been injected|
|CAVITY|patient anatomical cavity|
|BRACHY_CHANNEL|brachytherapy channel|
|BRACHY_ACCESSORY|brachytherapy accessory device|
|BRACHY_SRC_APP|brachytherapy source applicator|
|BRACHY_CHNL_SHLD|brachytherapy channel shield|
|SUPPORT|external patient support device|
|FIXATION|external patient fixation or immobilization device|
|DOSE_REGION|ROI to be used as a dose reference|
|CONTROL|ROI to be used in control of dose optimization and calculation|